# Instalación y Librerías

In [ ]:
# pip install ollama

In [ ]:
# Para tratamiento de datos
import pandas as pd
import numpy as np
import re #para llamar a Expresiones Regulares y estandarizar el nombre de las columnas.
import time
import requests

# Para visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns

# Para poder visualizar todas las columnas de los DataFrames
pd.set_option('display.max_columns', None) 

# Trabajar con el sistema operativo y variables de entorno
import os 
from dotenv import load_dotenv
load_dotenv() #carga las variables del entorno .env; devuelve un true o false
import ollama
MODELO_OLLAMA = 'gemma3:4b' 

# Conexión con MySQL
import mysql.connector
from mysql.connector import Error

# Gestión de los warnings
import warnings
warnings.filterwarnings("ignore")

#Gestión de las descargas
from tqdm import tqdm #barra progreso
from concurrent.futures import ThreadPoolExecutor, as_completed #paralelismo


# --- CONFIGURACIÓN ---
STEAM_KEY = os.getenv("steam_key")
MAX_THREADS_STEAM = 3  # Steam es estricto, no subir de 5-10 para evitar bloqueos
ARCHIVO_STEAM = "./datasets/dataset_steam.csv"
ARCHIVO_MAESTRO_JUEGOS = "./datasets/juegos_maestro.csv"
MAX_THREADS_OLLAMA = 4  # Ajusta según tu GPU/RAM (4-8 suele ser ideal)
PAUSA_STEAM = 1.0       # Respiro extra entre peticiones concurrentes



In [ ]:
# Cargamos las variables de entorno del archivo .env
load_dotenv()

# 1. Recuperamos la Key y el ID base
steam_key = os.getenv("steam_key")

# 2. Creamos la lista de los 16 IDs dinámicamente
lista_ids = [76561198056243081, 76561198842603734, 76561198023414915, 76561198048165534, 76561198254085126, 76561197984432884, 76561199080934614, 76561198294650349, 
             76561197986603983, 76561198212206651, 76561198062673538, 76561198014898339, 76561198067053149, 76561199499421434, 76561198046160451]

print(f"Se han cargado {len(lista_ids)} IDs.")

# DS Jugadores

In [ ]:
if not os.path.exists('./datasets'): os.makedirs('./datasets')

# --- FUNCIONES DE APOYO ---

def obtener_juegos_usuario(s_id):
    """Extrae la biblioteca de juegos de un SteamID."""
    url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        'key': STEAM_KEY,
        'steamid': s_id,
        'format': 'json',
        'include_appinfo': 1,
        'include_played_free_games': 1
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            juegos = response.json().get('response', {}).get('games', [])
            for j in juegos: j['user_id'] = s_id
            return juegos
        elif response.status_code == 429:
            print(f"\n⚠️ Rate limit alcanzado con ID {s_id}. Esperando...")
            time.sleep(10)
    except Exception:
        return []
    return []

def obtener_logros_worker(row_data):
    """Consulta los logros de un juego específico para un usuario."""
    s_id = row_data['user_id']
    a_id = row_data['appid']
    url = "https://api.steampowered.com/ISteamUserStats/GetPlayerAchievements/v0001/"
    params = {'key': STEAM_KEY, 'steamid': s_id, 'appid': a_id}
    
    try:
        r = requests.get(url, params=params, timeout=7)
        if r.status_code == 200:
            data = r.json()
            achievements = data.get('playerstats', {}).get('achievements', [])
            total = len(achievements)
            ganados = sum(1 for a in achievements if a.get('achieved') == 1)
            
            return {
                'achieved': ganados,
                'total_achievements': total,
                'completion_percentage': (ganados / total * 100) if total > 0 else 0
            }
    except Exception:
        pass
    
    return {'achieved': 0, 'total_achievements': 0, 'completion_percentage': 0}

# --- PROCESO PRINCIPAL ---

# 1. Validación de la lista global
# Asumimos que 'lista_ids' ya existe en tu entorno
ids_a_procesar = [i for i in lista_ids if i] # Limpieza rápida de nulos

# ─────────────────────────────────────────────────────────────
# PASO 1: Extracción de Bibliotecas
# ─────────────────────────────────────────────────────────────
print(f"--- PASO 1: Extrayendo juegos de {len(ids_a_procesar)} usuarios ---")
registros_completos = []

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(obtener_juegos_usuario, s_id) for s_id in ids_a_procesar]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Procesando Usuarios", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}"):
        registros_completos.extend(f.result())

if not registros_completos:
    print("❌ Error: No se obtuvieron datos. Verifica tu Steam Key y la lista de IDs.")
    exit()

df_steam = pd.DataFrame(registros_completos)

# ─────────────────────────────────────────────────────────────
# PASO 2: Muestreo Estratificado (25k registros)
# ─────────────────────────────────────────────────────────────
total_a_muestrear = min(25000, len(df_steam))
# Aseguramos que el cálculo de la muestra por usuario no sea cero
muestra_por_usuario = max(1, total_a_muestrear // len(ids_a_procesar))

df_sample = df_steam.groupby('user_id', group_keys=False).apply(
    lambda x: x.sample(min(len(x), muestra_por_usuario))
).sample(frac=1).reset_index(drop=True)

print(f"✅ Muestra lista: {len(df_sample)} juegos para enriquecer.")

# ─────────────────────────────────────────────────────────────
# PASO 3: Enriquecimiento de Logros (Paralelo)
# ─────────────────────────────────────────────────────────────
print(f"--- PASO 3: Obteniendo logros ({MAX_THREADS_STEAM} hilos) ---")
logros_resultados = [None] * len(df_sample) 

with ThreadPoolExecutor(max_workers=MAX_THREADS_STEAM) as executor:
    # Usamos un diccionario para rastrear el índice original
    futures = {executor.submit(obtener_logros_worker, row): i for i, row in df_sample.iterrows()}
    
    for f in tqdm(as_completed(futures), total=len(futures), desc="Consultando Logros", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}"):
        idx = futures[f]
        logros_resultados[idx] = f.result()

# ─────────────────────────────────────────────────────────────
# PASO 4: Unión y Limpieza Final
# ─────────────────────────────────────────────────────────────
df_extra = pd.DataFrame(logros_resultados)
df_final = pd.concat([df_sample, df_extra], axis=1)

# Limpieza de columnas innecesarias y formateo
cols_a_borrar = ['img_icon_url', 'has_leaderboards', 'playtime_windows_forever', 
                 'playtime_mac_forever', 'playtime_linux_forever', 'playtime_deck_forever']
df_final = df_final.drop(columns=cols_a_borrar, errors='ignore')
df_final["has_community_visible_stats"] = df_final["has_community_visible_stats"].fillna(False).astype(bool)

# Lógica para contenido adulto
df_final["content_descriptorids"] = df_final["content_descriptorids"].fillna(0)
df_final['adult_content'] = df_final['content_descriptorids'].apply(lambda x: x != 0 and x != "0")

df_final.to_csv(ARCHIVO_STEAM, index=False)
print(f"\n🚀 Proceso terminado. Archivo guardado: {ARCHIVO_STEAM}")

Procesando Usuarios: 100%|██████████| 15/15 [00:04<00:00,  3.23it/s]


✅ Muestra lista: 20700 juegos para enriquecer.
--- PASO 3: Obteniendo logros (3 hilos) ---


Consultando Logros: 100%|██████████| 20700/20700 [31:07<00:00, 11.08it/s] 


🚀 Proceso terminado. Archivo guardado: ./datasets/dataset_steam1.csv


# DS Contenido +18

In [ ]:
# 1. Cargamos el archivo
df_final = pd.read_csv("./datasets/dataset_steam.csv")

# 2. Diccionario de mapeo (Asegúrate de que content_id sean ENTEROS)
data_mapeo_contenido = {
    'content_id': [0, 1, 2, 3, 4, 5],
    'description_es': [
        'Contenido family friendly',
        'Contenido sexual explícito o desnudez',
        'Violencia frecuente o intensa',
        'Lenguaje fuerte',
        'Temas adultos/sensibles',
        'Escenas de sangre y desmembramiento'
    ]
}
df_mapeo = pd.DataFrame(data_mapeo_contenido)


# Convertimos toda la columna a un solo texto gigante para buscar números
todo_el_texto = ",".join(df_final['content_descriptorids'].fillna("0").astype(str))

# Usamos una Expresión Regular para sacar todos los números
ids_encontrados_str = re.findall(r'\d+', todo_el_texto)

# Convertimos a enteros y quitamos duplicados (usando un set)
unique_ids = {int(i) for i in ids_encontrados_str if i != '0'}

print(f"IDs detectados con éxito: {unique_ids}")

# 3. Filtramos el mapeo usando esos IDs
# También incluimos el 0 por defecto para que siempre haya una base
df_descriptors_unicos = df_mapeo[df_mapeo['content_id'].isin(list(unique_ids) + [0])].reset_index(drop=True)

# 4. Guardamos
df_descriptors_unicos.to_csv("./datasets/mapeo_contenido.csv", index=False)

print(f"✅ Archivo creado con {len(df_descriptors_unicos)} categorías reales.")
print(df_descriptors_unicos)

IDs detectados con éxito: {1, 2, 3, 4, 5}
✅ ¡Ahora sí! Archivo creado con 6 categorías reales.
   content_id                         description_es
0           0              Contenido family friendly
1           1  Contenido sexual explícito o desnudez
2           2          Violencia frecuente o intensa
3           3                        Lenguaje fuerte
4           4                Temas adultos/sensibles
5           5    Escenas de sangre y desmembramiento


# DS Maestro de juegos

In [7]:


if not os.path.exists('./datasets'): os.makedirs('./datasets')

# --- FUNCIONES DE APOYO ---

def obtener_info_juego(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 429:
            time.sleep(20)
            return None
        data = response.json()
        if data and data.get(str(appid), {}).get('success'):
            details = data[str(appid)]['data']
            genres_list = details.get('genres', [])
            date_str = details.get('release_date', {}).get('date', '')
            
            return {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': details.get('developers', ["Desconocido"])[0],
                'genres_steam': ", ".join([g['description'] for g in genres_list]) if genres_list else "Sin Género",
                'price': 0 if details.get('is_free') else (details.get('price_overview', {}).get('final', 0) / 100),
                'country': None,
                'genre': None,
                'release_year': int(pd.to_datetime(date_str, errors='coerce').year),
                'metacritic_score': details.get('metacritic', {}).get('score', 0),
                'total_recommendations': details.get('recommendations', {}).get('total', 0),
                'is_free': details.get('is_free', False)
            }
    except: return None

def obtener_pais_worker(dev):
    """Worker para procesar países en paralelo."""
    prompt = f"""Empresa: {dev}. Eres un buscador de sedes corporativas.
    Tarea: Indica el país donde se encuentra la sede principal de la empresa: '{dev}'.
    
    Reglas críticas de respuesta:
    1. Responde ÚNICAMENTE con el nombre del país en español.
    2. NO escribas frases como "La sede está en..." o "El país es...".
    3. NO uses puntos finales ni signos de puntuación.
    4. NO des ninguna información adicional, ni ciudades, ni fechas.
    5. Si no conoces el país, responde exclusivamente con la palabra: Desconocido.
    6. Responde con exactamente UNA palabra.
    """
    try:
        response = ollama.chat(model=MODELO_OLLAMA, messages=[{'role': 'user', 'content': prompt}])
        pais = response['message']['content'].strip().split('\n')[0].replace(".", "")
        return dev, pais
    except: return dev, "Error"

def obtener_genero_worker(combo_steam):
    """Clasifica un combo de etiquetas Steam en un género genérico."""
    generos_referencia = "Acción, RPG, Estrategia, Aventura, Simulación, Deportes, Puzzle, Shooter"
    
    # FIX: el parámetro es un combo de géneros Steam, no un nombre de juego
    prompt = f"""
    Eres un experto clasificador de videojuegos.
    Objetivo: A partir de estas etiquetas de Steam '{combo_steam}', determina el género principal más genérico.
    Reglas:
    1. Responde con exactamente UNA palabra.
    2. Género genérico, NO específico.
    3. Usa esta lista como preferencia: [{generos_referencia}].
    4. NO uses puntos ni explicaciones.
    """

    try:
        response = ollama.chat(model=MODELO_OLLAMA, messages=[{'role': 'user', 'content': prompt}])
        genero = response['message']['content'].strip().split('\n')[0].replace(".", "").capitalize()
        return combo_steam, genero
    except Exception as e:
        return combo_steam, "Error"

# --- PROCESO PRINCIPAL ---

# 1. Cargar o Crear Dataset Maestro
if os.path.exists(ARCHIVO_MAESTRO_JUEGOS):
    df_maestro = pd.read_csv(ARCHIVO_MAESTRO_JUEGOS)
else:
    df_maestro = pd.DataFrame(columns=['appid', 'name', 'developer', 'genres_steam', 'price', 'country', 'genre', 'release_year', 'metacritic_score', 'total_recommendations', 'is_free' ])

# ─────────────────────────────────────────────────────────────
# PASO 1: Steam
# ─────────────────────────────────────────────────────────────
df_inicial = pd.read_csv(ARCHIVO_STEAM)
ids_totales = df_inicial["appid"].unique()
ids_faltantes = [aid for aid in ids_totales if aid not in df_maestro['appid'].values]

if ids_faltantes:
    print(f"--- PASO 1: Steam ({len(ids_faltantes)} juegos, {MAX_THREADS_STEAM} hilos) ---")
    with ThreadPoolExecutor(max_workers=MAX_THREADS_STEAM) as executor:
        futures = {executor.submit(obtener_info_juego, aid): aid for aid in ids_faltantes}
        nuevos_registros = []
        for f in tqdm(as_completed(futures), total=len(futures), desc="Steam", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}"):
            res = f.result()
            if res:
                nuevos_registros.append(res)
            
            # Guardado incremental cada 20 resultados para no perder nada
            if len(nuevos_registros) >= 20:
                dfn = pd.DataFrame(nuevos_registros)
                if not dfn.empty:
                    # eliminar columnas totalmente vacías para evitar el FutureWarning
                    dfn = dfn.dropna(axis=1, how='all')
                    df_maestro = pd.concat([df_maestro, dfn], ignore_index=True)
                    df_maestro.to_csv(ARCHIVO_MAESTRO_JUEGOS, index=False)
                nuevos_registros = []
            time.sleep(PAUSA_STEAM / MAX_THREADS_STEAM)
        
        if nuevos_registros:
            dfn = pd.DataFrame(nuevos_registros)
            if not dfn.empty:
                dfn = dfn.dropna(axis=1, how='all')
                df_maestro = pd.concat([df_maestro, dfn], ignore_index=True)
                df_maestro.to_csv(ARCHIVO_MAESTRO_JUEGOS, index=False)

df_maestro.drop(columns=['country', 'genre'], inplace=True, errors='ignore')

# ─────────────────────────────────────────────────────────────
# PASO 2: Países
# ─────────────────────────────────────────────────────────────
print(f"\n--- PASO 2: Ollama Países ---")

devs_totales = df_maestro['developer'].unique()
print(f"Total de desarrolladores a procesar: {len(devs_totales)}")

with ThreadPoolExecutor(max_workers=MAX_THREADS_OLLAMA) as executor:
    futures = [executor.submit(obtener_pais_worker, d) for d in devs_totales]
    
    contador_pais = 0  # FIX: contador incremental real
    for f in tqdm(as_completed(futures), total=len(futures), desc="Países", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}"):
        dev, pais = f.result()
        df_maestro.loc[df_maestro['developer'] == dev, 'country'] = pais
        
        contador_pais += 1
        if contador_pais % 20 == 0: 
            df_maestro.to_csv(ARCHIVO_MAESTRO_JUEGOS, index=False)

# Guardado final Paso 2
df_maestro.to_csv(ARCHIVO_MAESTRO_JUEGOS, index=False)

# ─────────────────────────────────────────────────────────────
# PASO 3: Géneros
# ─────────────────────────────────────────────────────────────
print(f"\n--- PASO 3: Ollama Géneros ---")

combos_totales = df_maestro['genres_steam'].unique()
print(f"Total de combos únicos a reclasificar: {len(combos_totales)}")

with ThreadPoolExecutor(max_workers=MAX_THREADS_OLLAMA) as executor:
    futures = {executor.submit(obtener_genero_worker, c): c for c in combos_totales}
    
    contador_genero = 0
    for f in tqdm(as_completed(futures), total=len(futures), desc="Géneros", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}"):
        combo_original = futures[f]
        try:
            _, genero_limpio = f.result()
            df_maestro.loc[df_maestro['genres_steam'] == combo_original, 'genre'] = genero_limpio
            
            contador_genero += 1
            if contador_genero % 15 == 0:
                df_maestro.to_csv(ARCHIVO_MAESTRO_JUEGOS, index=False)
        except Exception as e:
            print(f"Error en combo {combo_original}: {e}")

# Guardado final absoluto
df_maestro.to_csv(ARCHIVO_MAESTRO_JUEGOS, index=False)
print(f"\n✅ Proceso completo. Datos en {ARCHIVO_MAESTRO_JUEGOS}")

--- PASO 1: Steam (12252 juegos, 3 hilos) ---


Steam: 100%|██████████| 12252/12252 [4:15:04<00:00,  1.25s/it]  



--- PASO 2: Ollama Países ---
Total de desarrolladores a procesar: 4662


Países: 100%|██████████| 4662/4662 [1:19:45<00:00,  1.03s/it]



--- PASO 3: Ollama Géneros ---
Total de combos únicos a reclasificar: 500


Géneros: 100%|██████████| 500/500 [06:22<00:00,  1.31it/s]


✅ Proceso completo. Datos en ./datasets/juegos_maestro1.csv
